# Spotify Data Cleaning: Daily Artists

* Simple pandas cleaning for `charts_artists_daily_365D.csv`. 
* The final DataFrame is `clean_artists_daily_365D`. 
* No intermediate CSV export. 
* The last cell loads into MySQL staging.
* Run cells in order. Original CSV goes in `data/raw/`.

---

## 1. Import and read source

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# Find the source file from the project folder.
def locate_raw_data(filename):
    for folder in (Path.cwd(), *Path.cwd().parents):
        candidate = folder / "data" / "raw" / filename
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"Missing CSV in data/raw/: {filename}")

artists_daily_df = pd.read_csv(locate_raw_data("charts_artists_daily_365D.csv"))
print("Source rows:", len(artists_daily_df))

Source rows: 4923038


---

## 2. Standardize source columns

In [3]:
# Retain consecutive_days, entry_rank, entry_date and peak_date for DW metrics.
artists_daily_df = artists_daily_df.rename(columns={"uri": "artist_uri"})

# Parse chart and historical milestone dates.
for column in ["date", "entry_date", "peak_date"]:
    if column in artists_daily_df:
        artists_daily_df[column] = pd.to_datetime(artists_daily_df[column], errors="coerce")

# Trim text fields and standardize country and entry-status values.
for column in ["country", "artist_name", "entry_status"]:
    if column in artists_daily_df:
        artists_daily_df[column] = artists_daily_df[column].astype("string").str.strip()
if "country" in artists_daily_df:
    artists_daily_df["country"] = artists_daily_df["country"].str.upper()
if "entry_status" in artists_daily_df:
    artists_daily_df["entry_status"] = artists_daily_df["entry_status"].str.replace("_", " ", regex=False)

# Remove the Spotify artist URI prefix while retaining the identifier.
artists_daily_df["artist_uri"] = (artists_daily_df["artist_uri"].astype("string")
    .str.strip().str.replace(r"^spotify:artist:", "", regex=True))

# Keep rank and longevity metrics as nullable integers.
for column in ["consecutive_days", "entry_rank", "peak_rank", "rank", "previous_rank", "days_on_chart", "streams"]:
    if column in artists_daily_df:
        artists_daily_df[column] = pd.to_numeric(artists_daily_df[column], errors="coerce").astype("Int64")


---

## 3. Handle missing values and duplicates

In [4]:
# One observation per chart key; artist master uses one row per artist URI.
keys = ["date", "country", "artist_uri"]
required = ["date", "country", "artist_uri", "rank"]
expected_columns = ['date', 'country', 'rank', 'artist_uri', 'artist_name', 'peak_rank', 'previous_rank', 'days_on_chart', 'consecutive_days', 'entry_status', 'peak_date', 'entry_rank', 'entry_date']
missing_columns = set(expected_columns) - set(artists_daily_df.columns)
if missing_columns:
    raise ValueError(f"Missing staging columns: {sorted(missing_columns)}")

# Standardize blank text. Keep optional unknown measures as SQL NULL.
for col in artists_daily_df.select_dtypes(include=["object", "string"]).columns:
    artists_daily_df[col] = artists_daily_df[col].astype("string").str.strip().replace("", pd.NA)

# Descriptive labels have meaningful defaults; unknown statistics do not.
for col, default in {'artist_name': 'Unknown Artist'}.items():
    artists_daily_df[col] = artists_daily_df[col].fillna(default)

# Invalid optional ranks/counts are unknown, not zero.
for col in ['peak_rank', 'entry_rank', 'previous_rank']:
    values = pd.to_numeric(artists_daily_df[col], errors="coerce")
    artists_daily_df[col] = values.where(values > 0).astype("Int64")
for col in ['days_on_chart', 'consecutive_days']:
    values = pd.to_numeric(artists_daily_df[col], errors="coerce")
    artists_daily_df[col] = values.where(values >= 0).astype("Int64")

# Required chart keys cannot be invented; keep only valid observations.
valid = artists_daily_df[required].notna().all(axis=1) & artists_daily_df["rank"].between(1, 200).fillna(False)

# Keep optional chart attributes within the staging constraints.
for col in ["peak_rank", "entry_rank"]:
    artists_daily_df[col] = artists_daily_df[col].where(artists_daily_df[col].between(1, 200))
artists_daily_df["consecutive_days"] = artists_daily_df["consecutive_days"].where(
    artists_daily_df["days_on_chart"].isna() | (artists_daily_df["consecutive_days"] <= artists_daily_df["days_on_chart"])
)

invalid_count = int((~valid).sum())
artists_daily_df = artists_daily_df.loc[valid]

# Fast single-pass grain deduplication. Keep first original source row.
duplicate_count = int(artists_daily_df.duplicated(subset=keys).sum())
clean_artists_daily_365D = artists_daily_df.drop_duplicates(subset=keys, keep="first").reset_index(drop=True)

print("Invalid key/rank rows removed:", invalid_count)
print("Repeated business keys removed:", duplicate_count)
print("Final staging rows:", len(clean_artists_daily_365D))


Invalid key/rank rows removed: 0
Repeated business keys removed: 39
Final staging rows: 4922999


---

## 4. Review and load to MySQL

In [5]:
print("DataFrame:", "clean_artists_daily_365D")
print("Rows:", len(clean_artists_daily_365D))
print("Missing values (optional columns):")
print(clean_artists_daily_365D.isna().sum().to_string())
display(clean_artists_daily_365D.head(5))

DataFrame: clean_artists_daily_365D
Rows: 4922999
Missing values (optional columns):
date                     0
country                  0
rank                     0
artist_uri               0
artist_name              0
peak_rank                0
previous_rank       232341
days_on_chart            0
consecutive_days         0
entry_status             0
peak_date                0
entry_rank               0
entry_date               0


,date,country,rank,artist_uri,artist_name,peak_rank,previous_rank,days_on_chart,consecutive_days,entry_status,peak_date,entry_rank,entry_date
0,2025-11-17,AE,1,3TVXtAsR1Inumwj472S9r4,Drake,1,2,1489,1489,MOVED UP,2021-10-21,1,2021-10-21
1,2025-11-17,AE,2,06HL4z0CvFAxyc27GXpf02,Taylor Swift,1,1,1489,1489,MOVED DOWN,2021-11-12,8,2021-10-21
2,2025-11-17,AE,3,1Xyo4u8uXC1ZmMpatF05PJ,The Weeknd,1,4,1489,1489,MOVED UP,2021-11-05,4,2021-10-21
3,2025-11-17,AE,4,4YRxDV8wJFPHPTeXepOstw,Arijit Singh,2,3,1489,1489,MOVED DOWN,2024-05-18,18,2021-10-21
4,2025-11-17,AE,5,1wRPtKGflJrBx9BmLsSwlU,Pritam,4,6,1489,1489,MOVED UP,2025-06-27,24,2021-10-21


In [6]:
# Import the shared staging loader from etl/load.py.
import sys
project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "etl" / "load.py").is_file()), None)
if project_root is None:
    raise FileNotFoundError("Missing etl/load.py in the project folder")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from etl.load import load_dataframe

In [7]:
# Load only the cleaned DataFrame; no exclusion-triggered ValueError.
if clean_artists_daily_365D.empty:
    raise ValueError("No valid records available for staging.")
load_dataframe(clean_artists_daily_365D, "clean_artists_daily_365D")
print("Loaded:", "clean_artists_daily_365D", "| Rows:", len(clean_artists_daily_365D))

Loaded 4,922,999 rows into spotify_analytics_stg.clean_artists_daily_365D
Loaded: clean_artists_daily_365D | Rows: 4922999
